# STARWOOD — real free-GPU open-source video benchmark

Run All in a **FREE** GPU notebook (prefer Kaggle Accelerator = **GPU T4 x2**; alternatively Colab free T4, less system RAM). This notebook generates real MP4 videos with **CogVideoX 2B** and **Wan 2.1 1.3B** from the same original prompt. Both listed checkpoints are Apache-2.0. No paid API, hosted video endpoint, credit purchases, fake frame generation or publishing. Check machine quotas first. A model failure is recorded, never labeled successful. The short low-resolution results are technical trials, not a feature-quality episode.

Official model cards: https://huggingface.co/zai-org/CogVideoX-2b and https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B

In [ ]:
import torch, os, json, sys, platform, time, gc, subprocess
from pathlib import Path
assert torch.cuda.is_available(), "Choose a FREE NVIDIA GPU runtime in Kaggle / Colab. CPU is not sufficient for these models."
GPUS=[{"name":torch.cuda.get_device_name(i),"GB":round(torch.cuda.get_device_properties(i).total_memory/2**30,2)} for i in range(torch.cuda.device_count())]
assert max(g["GB"] for g in GPUS)>=11, f"GPU memory too low for reliable evaluation: {GPUS}"
OUT=Path("/kaggle/working/starwood" if Path("/kaggle").exists() else "/content/starwood" if Path("/content").exists() else "starwood")
OUT.mkdir(parents=True,exist_ok=True)
print("FREE GPU",GPUS)
subprocess.run([sys.executable,"-m","pip","install","-q","--upgrade","diffusers","transformers","accelerate","ftfy","sentencepiece","safetensors","imageio[ffmpeg]","opencv-python-headless"],check=True)
from diffusers.utils import export_to_video
PROMPT=("A polished original 3D cinematic family adventure in a magical forest at blue hour. "
"Luma, an anthropomorphic red fox with expressive hazel eyes, rounded natural furry anatomy, turquoise scarf, "
"and Pip, an anthropomorphic cream rabbit with violet eyes and plum bow tie, stand together in a medium two-shot. "
"The northern star begins to fade. Luma blinks, turns her head, points with one paw, and looks worried. "
"Pip takes two physically grounded steps, looks upward, blinks and tilts his ears back in concern. "
"Elegant lifelike natural animal motion, smooth rounded bodies, realistic soft fur and flowing clothing, "
"high detail, rich saturated hues, soft cinematic lighting, expressive facial acting, no subtitles.")
NEG="low poly, faceted shapes, blur, grain, flicker, frozen frames, slideshow, extra limbs, deformed faces, robotic motion, logos, text"
SEED=20261026
RUN_MODELS=["cogvideox_2b","wan2_1_1_3b"]
print("READY",RUN_MODELS)

In [ ]:
from diffusers import CogVideoXPipeline, WanPipeline, AutoencoderKLWan
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
import traceback
def cog():
    p=CogVideoXPipeline.from_pretrained("zai-org/CogVideoX-2b",torch_dtype=torch.float16)
    p.enable_model_cpu_offload()
    p.vae.enable_slicing()
    p.vae.enable_tiling()
    frames=p(prompt=PROMPT,num_frames=49,num_inference_steps=35,guidance_scale=6.0,
             use_dynamic_cfg=True,generator=torch.Generator("cpu").manual_seed(SEED)).frames[0]
    dst=OUT/"STARWOOD_CogVideoX_2B.mp4"
    export_to_video(frames,str(dst),fps=8)
    del p,frames
    return dst
def wan():
    name="Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
    vae=AutoencoderKLWan.from_pretrained(name,subfolder="vae",torch_dtype=torch.float32)
    p=WanPipeline.from_pretrained(name,vae=vae,torch_dtype=torch.float16)
    p.scheduler=UniPCMultistepScheduler.from_config(p.scheduler.config,flow_shift=3.0)
    p.enable_model_cpu_offload()
    frames=p(prompt=PROMPT,negative_prompt=NEG,height=480,width=832,num_frames=81,
             num_inference_steps=30,guidance_scale=5.0,generator=torch.Generator("cpu").manual_seed(SEED)).frames[0]
    dst=OUT/"STARWOOD_Wan21_1_3B.mp4"
    export_to_video(frames,str(dst),fps=16)
    del p,vae,frames
    return dst
RECEIPTS=[]
for model in RUN_MODELS:
    start=time.time()
    try:
        path=({"cogvideox_2b":cog,"wan2_1_1_3b":wan}[model])()
        receipt={"model":model,"generated":True,"file":str(path),"bytes":path.stat().st_size,"seconds":round(time.time()-start,1)}
        print("REAL MP4 GENERATED",receipt)
    except Exception as e:
        receipt={"model":model,"generated":False,"error_type":type(e).__name__,"error":str(e)[:850],"seconds":round(time.time()-start,1)}
        print("FAILED; no output asserted",receipt); traceback.print_exc(limit=3)
    RECEIPTS.append(receipt); torch.cuda.empty_cache(); gc.collect()
(OUT/"render_receipts.json").write_text(json.dumps(RECEIPTS,indent=2))

In [ ]:
import cv2, numpy as np, csv, zipfile
RESULTS=[]
for info in RECEIPTS:
    result=dict(info)
    if info["generated"]:
        cap=cv2.VideoCapture(info["file"])
        fps=cap.get(cv2.CAP_PROP_FPS); count=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        previous=None; changes=[]; brightness=[];i=0
        while True:
            ok,im=cap.read()
            if not ok:break
            if i%max(1,round(fps/4))==0:
                g=cv2.cvtColor(cv2.resize(im,(160,90)),cv2.COLOR_BGR2GRAY)
                brightness.append(float(g.mean()))
                if previous is not None: changes.append(float(cv2.absdiff(g,previous).mean()))
                previous=g
            i+=1
        cap.release()
        result.update({"frame_count":count,"resolution":f"{w}x{h}","fps":fps,
           "duration":round(count/fps,2) if fps else 0,
           "mean_motion_delta":round(float(np.mean(changes)),3) if changes else 0,
           "static_fraction":round(float(np.mean(np.array(changes)<0.6)),3) if changes else 1,
           "mean_luma":round(float(np.mean(brightness)),2) if brightness else 0})
        result["technical_pass"]=result["duration"]>=4.5 and result["static_fraction"]<.85 and result["mean_luma"]>8
    else: result["technical_pass"]=False
    result["human_cinematic_review"]="NOT VERIFIED";result["character_continuity_review"]="NOT VERIFIED"
    RESULTS.append(result)
(OUT/"qa_results.json").write_text(json.dumps(RESULTS,indent=2))
with (OUT/"qa_results.csv").open("w",newline="") as f:
    keys=sorted({k for obj in RESULTS for k in obj})
    writer=csv.DictWriter(f,fieldnames=keys);writer.writeheader();writer.writerows(RESULTS)
with zipfile.ZipFile(OUT/"STARWOOD_REAL_MODEL_COMPARISON.zip","w",zipfile.ZIP_DEFLATED) as zipf:
    for file in OUT.iterdir():
        if file.is_file() and file.name!="STARWOOD_REAL_MODEL_COMPARISON.zip":zipf.write(file,arcname=file.name)
print(json.dumps(RESULTS,indent=2))
print("Download real results from:",OUT/"STARWOOD_REAL_MODEL_COMPARISON.zip")
assert any(x["generated"] for x in RECEIPTS),"No model completed. Inspect render_receipts.json."